## Python for Finance: Plotting Stock Market Data

In this tutorial we follow from import financial data to then graphing financial data in timeseries chart format with plotly candlestick charts.
Data source: https://plotly.com/python/candlestick-charts/

### Step 1: Import the dependencies

Ensure that plotly is installed. 

In [1]:
import pandas as pd
import datetime as dt
import plotly.offline as pyo
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf

pyo.init_notebook_mode(connected=True)

### Step 2: Get stock market data

Choose a date range and select stock to chart.

In [2]:
end = dt.datetime.now()
start = dt.datetime(2016, 1, 1)

df = yf.download('AMZN', start, end)

[*********************100%***********************]  1 of 1 completed


In [3]:
df.head()

Price,Close,High,Low,Open,Volume
Ticker,AMZN,AMZN,AMZN,AMZN,AMZN
Date,,,,,
2016-01-04,31.849501,32.886002,31.375500,32.814499,186290000
2016-01-05,31.689501,32.345501,31.382500,32.342999,116452000
2016-01-06,31.632500,31.989500,31.015499,31.100000,106584000
2016-01-07,30.396999,31.500000,30.260500,31.090000,141498000
2016-01-08,30.352501,31.207001,30.299999,30.983000,110258000


In [4]:
df = df.droplevel(level=1, axis=1)

In [5]:
df.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2016-01-04,31.849501,32.886002,31.375500,32.814499,186290000
2016-01-05,31.689501,32.345501,31.382500,32.342999,116452000
2016-01-06,31.632500,31.989500,31.015499,31.100000,106584000
2016-01-07,30.396999,31.500000,30.260500,31.090000,141498000
2016-01-08,30.352501,31.207001,30.299999,30.983000,110258000


### Step 3. Create Moving Average Terms

For this we use pandas rolling function and specify the rolling window parameter. Refs: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html. By choosing a rolling window of 50/200, inherently this requires 50/200 rows of data before it can compute the mean. We end up with NaN (Not a Number), we can resolve this (if we would like to have a continuous MA line throughout this period) by specifying the min_periods parameter.

In [6]:
df['MA50'] = df['Close'].rolling(window=50, min_periods=0).mean()
df['MA200'] = df['Close'].rolling(window=200, min_periods=0).mean()

In [7]:
len(df.iloc[:5, 5])

5

In [8]:
df['MA50'].head(10)

Date
2016-01-04    31.849501
2016-01-05    31.769501
2016-01-06    31.723834
2016-01-07    31.392125
2016-01-08    31.184200
2016-01-11    31.134667
2016-01-12    31.100357
2016-01-13    30.849125
2016-01-14    30.715889
2016-01-15    30.495200
Name: MA50, dtype: float64

In [9]:
df['MA200'].head()

Date
2016-01-04    31.849501
2016-01-05    31.769501
2016-01-06    31.723834
2016-01-07    31.392125
2016-01-08    31.184200
Name: MA200, dtype: float64

#### Step 4: Create plotly fig/subplot

In [10]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, subplot_titles=('AMZN', 'Volume'), row_width=[0.2, 0.8])

#### Step 5: Add Open High Low Close candle stick graph

In [11]:
fig.add_trace(go.Candlestick(x=df.index,
                            open=df['Open'],
                            high=df['High'],
                            low=df['Low'],
                            close=df['Close'],
                            name='OHLC'),
             row=1, col=1)
fig.show()

#### Step 6: Add Moving Average Terms

In [12]:
fig.add_trace(go.Scatter(x=df.index,
                        y=df['MA50'],
                        marker_color='grey',
                        name='MA50'),
             row=1, col=1)

fig.add_trace(go.Scatter(x=df.index,
                        y=df['MA200'],
                        marker_color='lightgrey',
                        name='MA200'),
             row=1, col=1)

#### Step 7: Add Volume Bar Chart in subplot

In [13]:
fig.add_trace(
    go.Bar(
        x=df.index,
        y=df['Volume'],
        marker_color='red',
        showlegend=False
    ),
    row=2, col=1
)

#### Step 8: Update layout with appropriate label, colors and sizes

In [14]:
fig.update_layout(
    title='AMZN Historical Price Chart',
    xaxis_tickfont_size = 12,
    yaxis = dict(
        title='Price ($/share)',
        title_font_size=14,
        tickfont_size=12
    ),
    autosize = False,
    width=800,
    height=500,
    margin=dict(l=50, r=50, b=100, t=100, pad=5),
    paper_bgcolor='LightSteelBlue'
)

#### Step 9: Finally, remove rangeslider from subplot

In [15]:
fig.update(layout_xaxis_rangeslider_visible=False)
fig.show()